In [33]:
import os
from pprint import pprint
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import psycopg
import pandas as pd
from typing import Any
import json
from psycopg import sql
import boto3
load_dotenv()
MODELO='gemini-3.1-flash-lite'
generador_consultas=ChatGoogleGenerativeAI(model=MODELO,temperature=0)
model=ChatGoogleGenerativeAI(model=MODELO)
parser=StrOutputParser()
import ast

In [34]:
consulta_messages=[('system',"""
        Lenguaje: sql pgAdmin
        Tabla: v_viviendas
        Columnas: [nombre, localidad (minusculas con _ en lugar de espacio, sin acentos, con nh en lugar de ñ), cp, precio, tipo (minusculas con _ en lugar de espacio, sin acentos, con nh en lugar de ñ), superficie_util, conservacion (A reformar, En buen estado, Reformado, A estrenar), superficie_solar, planta, habitaciones, banhos, referencia, antiguedad, superficie_construida, adaptado_a_personas_con_movilidad_reducida, balcon, se_aceptan_mascotas, calle_alumbrada, calle_asfaltada, lavadero, ascensor, chimenea, cocina_equipada, exterior, armarios_empotrados, garaje, sistema_de_seguridad, piscina, comedor, luz, carpinteria_exterior, orientacion, calefaccion, amueblado, urbanizado, trastero, puerta_blindada, vidrios_dobles, telefono, interior, soleado, agua, portero_automatico, aire_acondicionado, tipo_suelo, jardin, carpinteria_interior, terraza, gastos_de_comunidad, gas, precio_superficie, consumo, emisiones, descripcion]
        Tarea: Select de sql que dé lo que el usuario quiere (ten en cuenta las peticiones anteriores si falta información en la actual). NADA MÁS QUE ESO. SIN ADORNOS
        Salida: SELECT ... where ... is not null ...
        Importante: Eliminar los nulos en las columnas requeridas. Usa LIKE para columnas de texto y siempre con %. Usa la descripción solo si ayuda a responder la peticion. Que la consulta conteste a la peticion con las mínimas filas necesarias (100 como máximo).
    """)]
def sacar_consulta(peticion:str)->str:
    global consulta_messages
    consulta_messages.append(('human','{request}'))
    prompt=ChatPromptTemplate.from_messages(messages=consulta_messages)
    response=' '.join(((prompt | generador_consultas | parser).invoke(input={"request":peticion})).replace(';','').replace('sql ','').replace('"',"'").split())
    consulta_messages.append(('ai',response))
    return response

In [36]:
json.dumps([('ai','aaaaa')])

'[["ai", "aaaaa"]]'

In [35]:
messages=[('system','ERES ANALISTA DEL MERCADO INMOBILIARIO DE MADRID. NO TE PRESENTES. NO ENSEÑES DIRECTAMENTE LOS ANUNCIOS DE REFERENCIA. CÍÑETE SOBRETODO A LOS DATOS QUE SE TE PROPORCIONA. HABLA DE LOS DATOS COMO SI NO FUERA EL USUARIO QUIEN TE LOS DIÓ. USA LAS DESCRIPCIONES SI TE ES ÚTIL. TODOS LOS PISOS ESTÁN EN VENTA.')]
while True:
    request=input('User (introduce esc para salir): ')
    if request.lower() in ['esc']:
        break
    q=sacar_consulta(request)
    print(q)
    query=sql.SQL("select row_to_json(t) from ({subquery}) t;").format(
        subquery=sql.SQL(q)
    )
    async with await psycopg.AsyncConnection.connect(str(os.environ.get('URL'))) as connection:
        async with connection.cursor() as cursor:
            await cursor.execute(query)
            filas=await cursor.fetchall()
            resultado=[fila for fila in filas]
    messages.extend([('ai',f'ANUNCIOS DE REFERENCIA: {json.dumps(resultado).replace('{','{{').replace('}','}}')}'),('human','{request}')])
    prompt_chat=ChatPromptTemplate.from_messages(messages=messages)
    cadena=prompt_chat | model | parser
    respuesta=cadena.astream(input={"request":request})
    completo:str=''
    async for chunk in respuesta:
        completo+=chunk
        print(chunk,end='',flush=True)
    messages.append(('ai',completo))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


SELECT nombre, localidad, precio, referencia FROM v_viviendas WHERE precio IS NOT NULL ORDER BY precio DESC LIMIT 100


Direct use of automatic function calling (AFC) in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream. Similarly, direct use of AFC in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message.


Basándome en los datos del mercado actual, el segmento de precios más altos, que se sitúa en torno a los **3.000.000 de euros**, se concentra principalmente en ubicaciones estratégicas de gran prestigio.

Los puntos donde se encuentran las propiedades con los precios más elevados son:

*   **Chamartín (Ciudad Jardín):** Encabeza la lista con propiedades que alcanzan los 3.000.000 €.
*   **Recoletos:** Es una de las zonas con mayor densidad de oferta en este rango de precios, con múltiples activos situados en los 2.990.000 € y 2.995.000 €.
*   **Trafalgar:** Registra activos de gran valor en torno a los 2.990.000 €.
*   **Nueva España:** Presenta opciones de alta gama, como dúplex, valorados en 2.990.000 €.
*   **Justicia-Chueca:** Se mantiene en este mismo escalón de precios, alcanzando los 2.990.000 €.

Es posible observar una tendencia clara donde el grueso de la oferta de ultra-lujo en Madrid se aglutina en los barrios de **Salamanca (con especial fuerza en Recoletos y Goya)** y zon